In [1]:
#Delete this
# 
# testing kernel
#import sys
#print(sys.executable)


In [2]:
#DELETE THIS:
# first run in terminal:
#   pip install transformers accelerate torch numpy evaluate datasets scikit-learn

In [3]:
from pathlib import Path
from datasets import load_dataset, ClassLabel, Dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer, set_seed
import torch
import evaluate
import numpy as np
import pandas as pd
#from datasets import Dataset
from sklearn.utils import resample
from collections import Counter
from sklearn.metrics import classification_report
from torch.nn import CrossEntropyLoss
from transformers import Trainer
import transformers

 

/work/nlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
set_seed(1802)

In [5]:
# setting path working directory

path = Path.cwd()

data_path = path.parents[0] / "my-repo" / "Exam data" #setting path to data folder

csv_train = data_path / "train.csv" #loading UCC train dataset csv
csv_test = data_path / "test.csv"  # UCC test dataset
csv_val = data_path / "val.csv"     #UCC val



In [6]:
#read csv
df_train = pd.read_csv(csv_train)
df_test = pd.read_csv(csv_test)
df_val = pd.read_csv(csv_val)

#turn into hugging face ds
ds_train_og = Dataset.from_pandas(df_train)  #original train 
ds_test_og = Dataset.from_pandas(df_test)
ds_val = Dataset.from_pandas(df_val)


In [7]:
#print if ds 
print(ds_train_og)
print(ds_test_og)
print(ds_val)

Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence'],
    num_rows: 35503
})
Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence'],
    num_rows: 4425
})
Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:conf

In [8]:
ds_train_og[:5]

{'_unit_id': [2319157561, 1739464982, 1739457583, 2319156950, 2327196492],
 '_trusted_judgments': [4, 4, 5, 40, 3],
 'comment': ['Three marriages, several bankrupt periods, inherited wealth, so whats your point?? For some its called life!',
  "The sense of entitlement among high school 'jocks' is boundless. Says this former high school nerd.",
  'So what? He was just stating the obvious.',
  'If one is a Con, why yes, one would honk. Loudly, repeatedly and in as coarse a manner as possible.',
  "Ooohhh... It's Wendy Whiner... making sure to ridicule males at even the hint that someone suggests they are victims."],
 'antagonize': [0, 0, 0, 0, 0],
 'antagonize:confidence': [1.0, 0.7634, 0.8121, 0.8508, 1.0],
 'condescending': [0, 0, 0, 0, 0],
 'condescending:confidence': [1.0, 0.7634, 0.5928, 0.8867, 1.0],
 'dismissive': [0, 0, 0, 0, 0],
 'dismissive:confidence': [1.0, 0.7634, 0.8043, 0.9239, 1.0],
 'generalisation': [0, 0, 0, 0, 0],
 'generalisation:confidence': [1.0, 1.0, 1.0, 0.8863, 

In [9]:
# print example
print(ds_train_og["comment"][12]) #remember zero indexed :)
print(ds_val["comment"][1])


But I see you two gentleman are not up to a intelligent rebuttal
Should we really pick a voting system that ensures the Liberals will never be removed from government? That doesn't sound like democracy.


# Preprocessing

In [10]:
#checking columns names are the same in train and original test ds
print(ds_train_og.column_names)
print(ds_test_og.column_names)

['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence']
['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence']


In [11]:
#merging train and test dataset (to get more data as we will not be using this test dataset)
ds_train = concatenate_datasets([ds_train_og, ds_test_og])

print(ds_train)
print(ds_test_og)
print(ds_val)


Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence'],
    num_rows: 39928
})
Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence'],
    num_rows: 4425
})
Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:conf

In [12]:
#removing empty comments ("")
def has_comment(example):
    return example["comment"] is not None and example["comment"] != ""

ds_train = ds_train.filter(has_comment)
ds_val = ds_val.filter(has_comment)

Filter: 100%|██████████| 4427/4427 [00:00<00:00, 73776.07 examples/s]


In [13]:
print(ds_train)
print(ds_val)

Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence'],
    num_rows: 39927
})
Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence'],
    num_rows: 4427
})


## Data wrangling
### creating new column where unhealthy columns cannot also be healthy
or more correct: comments with 1 in both healhty and min one unhealhty are now counted as unhealhty instead of healhty, as if we had used the healhty columns

In [14]:
#all types of unhealthy 
unhealthy_cols = [
    "antagonize",
    "condescending",
    "dismissive",
    "generalisation",
    "generalisation_unfair",
    "hostile",
    "sarcastic"
]

In [15]:
# add 1 in new unhealthy column if one of the 6 unhealthy columns contain a 1
# for train
def unhealthy(comment):
    comment["unhealthy"] = int(any(comment[col] == 1 for col in unhealthy_cols))
    return comment

# add to train
ds_train = ds_train.map(unhealthy)

print(ds_train[:20])
print(ds_train)

Map: 100%|██████████| 39927/39927 [00:07<00:00, 5256.08 examples/s]

{'_unit_id': [2319157561, 1739464982, 1739457583, 2319156950, 2327196492, 1739445471, 1739459230, 1739444194, 1739449016, 1739471591, 1739446352, 1739444207, 1800633246, 2028122642, 2327178861, 1739471243, 1739447511, 1739441445, 2327197661, 1739471701], '_trusted_judgments': [4, 4, 5, 40, 3, 3, 3, 5, 3, 3, 3, 3, 5, 5, 5, 3, 3, 6, 3, 3], 'comment': ['Three marriages, several bankrupt periods, inherited wealth, so whats your point?? For some its called life!', "The sense of entitlement among high school 'jocks' is boundless. Says this former high school nerd.", 'So what? He was just stating the obvious.', 'If one is a Con, why yes, one would honk. Loudly, repeatedly and in as coarse a manner as possible.', "Ooohhh... It's Wendy Whiner... making sure to ridicule males at even the hint that someone suggests they are victims.", 'anti-Harper rant....**YAWN**', "You seem to be suggesting that, because a majority government is going to get its way, it's never worth the effort to criticize a m

In [16]:
#Same for val

# add 1 in unhealthy column if one of the 6 unhealthy columns contain a 1
def unhealthy(comment):
    comment["unhealthy"] = int(any(comment[col] == 1 for col in unhealthy_cols))
    return comment

# add to val
ds_val = ds_val.map(unhealthy)


print(ds_val[:10])
print(ds_val)

Map: 100%|██████████| 4427/4427 [00:00<00:00, 7154.32 examples/s]


{'_unit_id': [1739468038, 1739467038, 1739449108, 1739450008, 1739444458, 1739456118, 1739461908, 1739449088, 1739461078, 2327215068], '_trusted_judgments': [3, 3, 3, 5, 5, 4, 5, 5, 3, 5], 'comment': ["That's exactly what they've done. Beck, Toronto's most popular taxi company, offers an app. All their taxis are licensed taxis with taxi plates affixed.", "Should we really pick a voting system that ensures the Liberals will never be removed from government? That doesn't sound like democracy.", "actually, the president of the united states has his finger on the nuclear button. once pressed, it's rather difficult to 'overturn' that kind of decision.", 'There is no aura of power, there only is irresponsible self-indulgence, if you serve the public and abuse taxpayer funded money, perks and responsibility.', 'Delusional much? I think you had better do some serious research before posting such nonsense.', 'I recall this argument before when the Vatican argued that child sex abuse by priests 

### checking balance for "unhealthy" column

In [17]:
#checking balance for unhealthy/healhty (BEFORE downsampling)
#1 = healhty
#0 = unhealhty

print(Counter(ds_train["unhealthy"]))
print(Counter(ds_val["unhealthy"]))

Counter({0: 35323, 1: 4604})
Counter({0: 3917, 1: 510})


#### Just extra overview

In [18]:
#how many of each type of comment

print("Healthy:", sum(ds_train["healthy"]))
print("unhealthy", sum(ds_train["unhealthy"]))
print("Antagonize:", sum(ds_train["antagonize"]))
print("Condescending:", sum(ds_train["condescending"]))
print("Dismissive:", sum(ds_train["dismissive"]))
print("Generalisation:", sum(ds_train["generalisation"]))
print("Generalisation_unfair:", sum(ds_train["generalisation_unfair"]))
print("Hostile:", sum(ds_train["hostile"]))
print("Sarcastic:", sum(ds_train["sarcastic"]))


#seeing the same number of unhealhty but a bigger number of healhty as those that are both healhty and a type of unhealhty were only counted as unhealhty before


Healthy: 36952
unhealthy 4604
Antagonize: 1892
Condescending: 2196
Dismissive: 1221
Generalisation: 848
Generalisation_unfair: 802.0
Hostile: 1031
Sarcastic: 1702


In [19]:
print("Healthy:", sum(ds_val["healthy"]))
print("unhealthy", sum(ds_val["unhealthy"]))
print("Antagonize:", sum(ds_val["antagonize"]))
print("Condescending:", sum(ds_val["condescending"]))
print("Dismissive:", sum(ds_val["dismissive"]))
print("Generalisation:", sum(ds_val["generalisation"]))
print("Generalisation_unfair:", sum(ds_val["generalisation_unfair"]))
print("Hostile:", sum(ds_val["hostile"]))
print("Sarcastic:", sum(ds_val["sarcastic"]))

Healthy: 4091
unhealthy 510
Antagonize: 174
Condescending: 238
Dismissive: 143
Generalisation: 96
Generalisation_unfair: 88.0
Hostile: 99
Sarcastic: 195


In [20]:
# how many comments are rated as both healthy and minimum one type of unhealthy
num_both_train = sum(1 for row in ds_train if row["healthy"] == 1 and row["unhealthy"] == 1)
num_both_val = sum(1 for row in ds_val if row["healthy"] == 1 and row["unhealthy"] == 1)


print(num_both_train)
print(num_both_val)

2622
287


## Downsampling

In [21]:
#splitting ds into majority and minority

train_majority = ds_train.filter(lambda x: x["unhealthy"] == 0) # healhty comments
train_minority = ds_train.filter(lambda x: x ["unhealthy"] == 1) # unhealhty comments

val_majority = ds_val.filter(lambda x: x ["unhealthy"] == 0) 
val_minority = ds_val.filter(lambda x: x ["unhealthy"] == 1)

Filter: 100%|██████████| 4427/4427 [00:00<00:00, 107824.79 examples/s]


In [22]:
#converting to pandas

train_majority_df = train_majority.to_pandas()
train_minority_df = train_minority.to_pandas()

val_majority_df = val_majority.to_pandas()
val_minority_df = val_minority.to_pandas()


In [23]:
#actual downsamling, to the length of min(unhealhty)

#train
train_majority_downsampled = resample(
    train_majority_df, 
    replace=False,                      # wihtout replacement
    n_samples=len(train_minority_df),  # match length of minority
    random_state=1802             
)

#val
val_majority_downsampled = resample(
    val_majority_df,
    replace=False,
    n_samples=len(val_minority_df),
    random_state=1802
)


In [24]:
#combine min and maj
train_balanced_df = pd.concat([train_minority_df, train_majority_downsampled])
val_balanced_df   = pd.concat([val_minority_df, val_majority_downsampled])



In [25]:
#checking balance for unhealthy/healhty (AFTER downsampling)

print("Train balanced counts:")
print(train_balanced_df["unhealthy"].value_counts())

print("Val balanced counts:")
print(val_balanced_df["unhealthy"].value_counts())


Train balanced counts:
unhealthy
1    4604
0    4604
Name: count, dtype: int64
Val balanced counts:
unhealthy
1    510
0    510
Name: count, dtype: int64


In [26]:
#converting back to huggingface ds
train_balanced_ds = Dataset.from_pandas(train_balanced_df)
val_balanced_ds = Dataset.from_pandas(val_balanced_df)

# shuffle
train_balanced_ds = train_balanced_ds.shuffle(seed=1802)
val_balanced_ds = val_balanced_ds.shuffle(seed=1802)


print(train_balanced_ds)
print(val_balanced_ds)

Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence', 'unhealthy', '__index_level_0__'],
    num_rows: 9208
})
Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence', 'unhealthy', '__index_level_0__'],
    num_rows: 1020
})


In [27]:
#removing the new index column
train_balanced_ds = train_balanced_ds.remove_columns("__index_level_0__")
val_balanced_ds   = val_balanced_ds.remove_columns("__index_level_0__")

print(train_balanced_ds)
print(val_balanced_ds)

Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence', 'unhealthy'],
    num_rows: 9208
})
Dataset({
    features: ['_unit_id', '_trusted_judgments', 'comment', 'antagonize', 'antagonize:confidence', 'condescending', 'condescending:confidence', 'dismissive', 'dismissive:confidence', 'generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence', 'hostile', 'hostile:confidence', 'sarcastic', 'sarcastic:confidence', 'unhealthy'],
    num_rows: 1020
})


# Using weights

In [28]:
'''
#finding the weights from the distribution of healthy/unhealthy

# number of healhty and unhealhty
n_samples = [35323, 4604]  # healthy, unhealthy

# Compute weights, proportional, inversely
weights = [sum(n_samples)/n for n in n_samples]  

# Convert to tensor
class_weights = torch.tensor(weights).to("cuda" if torch.cuda.is_available() else "cpu")
'''

'\n#finding the weights from the distribution of healthy/unhealthy\n\n# number of healhty and unhealhty\nn_samples = [35323, 4604]  # healthy, unhealthy\n\n# Compute weights, proportional, inversely\nweights = [sum(n_samples)/n for n in n_samples]  \n\n# Convert to tensor\nclass_weights = torch.tensor(weights).to("cuda" if torch.cuda.is_available() else "cpu")\n'

In [29]:
'''
def weighted_loss(model, inputs):
    labels = inputs["labels"]
    outputs = model(**inputs)
    logits = outputs.logits

    loss_fn = CrossEntropyLoss(weight=class_weights)
    loss = loss_fn(logits, labels)

    return loss
    '''

'\ndef weighted_loss(model, inputs):\n    labels = inputs["labels"]\n    outputs = model(**inputs)\n    logits = outputs.logits\n\n    loss_fn = CrossEntropyLoss(weight=class_weights)\n    loss = loss_fn(logits, labels)\n\n    return loss\n    '

In [30]:
'''
#changing names to fit same code as for balanced dataset

train_balanced_ds = ds_train
val_balanced_ds = ds_val
'''

'\n#changing names to fit same code as for balanced dataset\n\ntrain_balanced_ds = ds_train\nval_balanced_ds = ds_val\n'

In [31]:
'''
print(train_balanced_ds)
print(val_balanced_ds)
'''

'\nprint(train_balanced_ds)\nprint(val_balanced_ds)\n'

# MORE WEIGHTS STUFF (ROBERTA)


In [32]:
'''
from torch import nn
from transformers import RobertaForSequenceClassification, Trainer, TrainingArguments
'''

'\nfrom torch import nn\nfrom transformers import RobertaForSequenceClassification, Trainer, TrainingArguments\n'

In [33]:

'''
class WeightedRoberta(RobertaForSequenceClassification):
    def forward(self, labels=None, **kwargs):
        outputs = super().forward(**kwargs)
        if labels is not None:
            outputs.loss = nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
        return outputs
'''

'\nclass WeightedRoberta(RobertaForSequenceClassification):\n    def forward(self, labels=None, **kwargs):\n        outputs = super().forward(**kwargs)\n        if labels is not None:\n            outputs.loss = nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)\n        return outputs\n'

In [34]:
'''
model = WeightedRoberta.from_pretrained("roberta-base", num_labels=2)
model.to(device)

'''

'\nmodel = WeightedRoberta.from_pretrained("roberta-base", num_labels=2)\nmodel.to(device)\n\n'

# RoBERTa/BERT classifier

In [36]:
#changing the unhealhty column name to labels

train_balanced_ds = train_balanced_ds.rename_column("unhealthy", "labels")
val_balanced_ds = val_balanced_ds.rename_column("unhealthy", "labels")

In [37]:
# defining column and classes
num_classes = 2

train_balanced_ds = train_balanced_ds.cast_column("labels", ClassLabel(num_classes=num_classes))
val_balanced_ds = val_balanced_ds.cast_column("labels", ClassLabel(num_classes=num_classes))

Casting the dataset: 100%|██████████| 1020/1020 [00:00<00:00, 51062.75 examples/s]


## Splitting into train and val

In [ ]:
'''
Delete, already split
ds_downsampled = ds.train_test_split(train_size=2000,test_size=500, seed=42, stratify_by_column="label")
train_ds = ds_downsampled["train"]
val_ds = ds_downsampled["test"]
'''

## 1.2 loading the model

In [38]:
# define model + where to load it from (if already downloaded/cached)
model_path = path.parents[0] / "models" / "hf"
model_id = "roberta-base" #"distilbert/distilbert-base-uncased"

# GPU or CPU? Default to CPU if no GPU available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


# NFS: i have changed cased to uncased above^^^^

In [39]:
model = AutoModelForSequenceClassification.from_pretrained(
                                                            model_id, 
                                                            num_labels=num_classes, # pre-defined number of labels in dataset
                                                            cache_dir=model_path, 
                                                           ).to(device) # move model to device (cpu or gpu)
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=model_path)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#printing parameters
print(model.num_parameters())

## 1.3 tokenization

In [40]:
#defining function

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["comment"], truncation=True)

In [41]:
tokenized_train = train_balanced_ds.map(preprocess_function, batched=True)
tokenized_val = val_balanced_ds.map(preprocess_function, batched=True)

Map: 100%|██████████| 1020/1020 [00:00<00:00, 19844.84 examples/s]


## 1.4 training configuration

In [42]:
batch_size = 16
n_epochs = 2  # 2 or 3 epochs seemed to make prediction/loss/F1 worse or not better, therefore back to 1(on OG data, not merge)
output_dir = path.parents[0] / "training" / "distilbert_unhealthy_comments"

#defining training arguments
training_args = TrainingArguments(
   output_dir=output_dir,
   learning_rate=2e-5,                       #og 2e-5
   per_device_train_batch_size=batch_size,
   per_device_eval_batch_size=batch_size,
   num_train_epochs=n_epochs,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none", 
   #warmup_ratio = 0.06
)

In [43]:
#adding padding variable
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [44]:
def compute_metrics(eval_pred):
    """Take predicted logits and true labels to compute F1 score"""
    logits, labels = eval_pred

    # convert to predicted class label (index of highest logit)
    predictions = np.argmax(logits, axis=-1)

    # load and compute F1 from "evaluate" library
    f1_metric = evaluate.load("f1")
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"  # or "weighted"
    )["f1"]

    precision_metric = evaluate.load("precision")
    precision = precision_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"  # or "weighted"
    )["precision"]

    recall_metric = evaluate.load("recall")
    recall = recall_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"  # or "weighted"
    )["recall"]


    accuracy_metric = evaluate.load("accuracy")
    accuracy = accuracy_metric.compute(
        predictions=predictions, 
        references=labels
    )["accuracy"]
    
    return {"f1": f1, "precision": precision, "recall": recall, "accuracy": accuracy}

# 1.5 train and evaluate

In [ ]:
print(tokenized_train)

In [45]:
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_val,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

/tmp/ipykernel_172092/2519551727.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

## confusion matrix

# prediction report

In [ ]:
#make predictions on val ds
predictions = trainer.predict(tokenized_val)

# Get raw logits from trainer predictions
logits = predictions.predictions

# Convert logits to predicted class indices
y_pred = np.argmax(logits, axis=-1)

# Get true labels
y_true = predictions.label_ids

In [ ]:
classification_report_metric = evaluate.load("bstrai/classification_report") #loading .... whatever is gonna make the report 

results = classification_report_metric.compute(references=y_true, predictions=y_pred) #taking the true and predicted values

report_df = pd.DataFrame(results).transpose()#.round(2) #turning it into a pd to print the results neatly

print(report_df)

## with weights
{'eval_loss': 0.3213966488838196,
 'eval_f1': 0.4965324435424835,
 'eval_precision': 0.6579332552693208,
 'eval_recall': 0.5121529081379808,
 'eval_accuracy': 0.883668398463971,
 'eval_runtime': 9.5369,
 'eval_samples_per_second': 464.198,
 'eval_steps_per_second': 23.278,
 'epoch': 2.0}

precision    recall  f1-score      support
0              0.887295  0.994894  0.938019  3917.000000
1              0.428571  0.029412  0.055046   510.000000
accuracy       0.883668  0.883668  0.883668     0.883668
macro avg      0.657933  0.512153  0.496532  4427.000000
weighted avg   0.834449  0.883668  0.836299  4427.000000

# Changing to roBERTa

### BEST ONE SO FAR 2 epochs + batch size 16 

{'eval_loss': 0.6036847233772278,
 'eval_f1': 0.6955118328608406,
 'eval_precision': 0.6975488449202832,
 'eval_recall': 0.696078431372549,
 'eval_accuracy': 0.696078431372549,
 'eval_runtime': 4.6505,
 'eval_samples_per_second': 219.333,
 'eval_steps_per_second': 13.762,
 'epoch': 2.0}

Report
               precision    recall  f1-score      support
0              0.714592  0.652941  0.682377   510.000000
1              0.680505  0.739216  0.708647   510.000000
accuracy       0.696078  0.696078  0.696078     0.696078
macro avg      0.697549  0.696078  0.695512  1020.000000
weighted avg   0.697549  0.696078  0.695512  1020.000000


### NOT THIS 2 epochs + batch size 16 + warm up = 0.06              WORSE IN ALL THAN ABOVE

{'eval_loss': 0.9024559855461121,
 'eval_f1': 0.6771527718317285,
 'eval_precision': 0.6781090457244954,
 'eval_recall': 0.6774509803921569,
 'eval_accuracy': 0.6774509803921569,
 'eval_runtime': 4.5982,
 'eval_samples_per_second': 221.825,
 'eval_steps_per_second': 13.918,
 'epoch': 2.0}

 report
                precision    recall  f1-score      support
0              0.688935  0.647059  0.667341   510.000000
1              0.667283  0.707843  0.686965   510.000000
accuracy       0.677451  0.677451  0.677451     0.677451
macro avg      0.678109  0.677451  0.677153  1020.000000
weighted avg   0.678109  0.677451  0.677153  1020.000000


### LR 1e-5                                                         WORSE IN ALL THAN the top one

{'eval_loss': 1.0440375804901123,
 'eval_f1': 0.6811147804020743,
 'eval_precision': 0.6819608962466106,
 'eval_recall': 0.6813725490196079,
 'eval_accuracy': 0.6813725490196079,
 'eval_runtime': 4.669,
 'eval_samples_per_second': 218.46,
 'eval_steps_per_second': 10.923,
 'epoch': 2.0}

              precision    recall  f1-score      support
0              0.692308  0.652941  0.672048   510.000000
1              0.671614  0.709804  0.690181   510.000000
accuracy       0.681373  0.681373  0.681373     0.681373
macro avg      0.681961  0.681373  0.681115  1020.000000
weighted avg   0.681961  0.681373  0.681115  1020.000000

## Playing with learning rate

### change xx: epochs 1 + batch size 16 + LR 1e-

{'eval_loss': 0.608317494392395,
 'eval_f1': 0.6693761776562008,
 'eval_precision': 0.6700845507015047,
 'eval_recall': 0.669607843137255,
 'eval_accuracy': 0.6696078431372549,
 'eval_runtime': 4.936,
 'eval_samples_per_second': 206.646,
 'eval_steps_per_second': 12.966,
 'epoch': 1.0}

report:
              precision    recall  f1-score      support
0              0.679089  0.643137  0.660624   510.000000
1              0.661080  0.696078  0.678128   510.000000
accuracy       0.669608  0.669608  0.669608     0.669608
macro avg      0.670085  0.669608  0.669376  1020.000000
weighted avg   0.670085  0.669608  0.669376  1020.000000

### epochs 1 + batch size 16 + LR 2e-5 + weightdecay = 0.05

{'eval_loss': 0.6058403253555298,
 'eval_f1': 0.677885226393077,
 'eval_precision': 0.6796497584541062,
 'eval_recall': 0.6784313725490196,
 'eval_accuracy': 0.6784313725490196,
 'eval_runtime': 4.0462,
 'eval_samples_per_second': 252.089,
 'eval_steps_per_second': 15.817,
 'epoch': 1.0}

Report:
                precision    recall  f1-score      support
0              0.694444  0.637255  0.664622   510.000000
1              0.664855  0.719608  0.691149   510.000000
accuracy       0.678431  0.678431  0.678431     0.678431
macro avg      0.679650  0.678431  0.677885  1020.000000
weighted avg   0.679650  0.678431  0.677885  1020.000000


### 2 epoch + above (+ batch size 16 + LR 2e-5 + weightdecay = 0.05)

{'eval_loss': 1.1545531749725342,
 'eval_f1': 0.6705426356589147,
 'eval_precision': 0.6706827309236947,
 'eval_recall': 0.6705882352941177,
 'eval_accuracy': 0.6705882352941176,
 'eval_runtime': 3.9573,
 'eval_samples_per_second': 257.751,
 'eval_steps_per_second': 16.173,
 'epoch': 2.0}

Report:
  precision    recall  f1-score      support
0              0.674699  0.658824  0.666667   510.000000
1              0.666667  0.682353  0.674419   510.000000
accuracy       0.670588  0.670588  0.670588     0.670588
macro avg      0.670683  0.670588  0.670543  1020.000000
weighted avg   0.670683  0.670588  0.670543  1020.000000

### OG arguments:

trainer.evaluate():

{'eval_loss': 0.8903047442436218,
 'eval_f1': 0.6537037037037037,
 'eval_precision': 0.6570760233918129,
 'eval_recall': 0.6549019607843137,
 'eval_accuracy': 0.6549019607843137,
 'eval_runtime': 5.2718,
 'eval_samples_per_second': 193.482,
 'eval_steps_per_second': 24.28,
 'epoch': 1.0}

Report:
              precision    recall  f1-score      support
0              0.675556  0.596078  0.633333   510.000000
1              0.638596  0.713725  0.674074   510.000000
accuracy       0.654902  0.654902  0.654902     0.654902
macro avg      0.657076  0.654902  0.653704  1020.000000
weighted avg   0.657076  0.654902  0.653704  1020.000000

 

### Change 1:  2 epochs

trainer.evaluate():

{'eval_loss': 1.8954190015792847,                                   !! loss higher :((
 'eval_f1': 0.6449615384615385,
 'eval_precision': 0.6453215248363496,
 'eval_recall': 0.6450980392156862,
 'eval_accuracy': 0.6450980392156863,
 'eval_runtime': 4.1395,
 'eval_samples_per_second': 246.409,
 'eval_steps_per_second': 30.922,
 'epoch': 2.0}

Report: 
               precision    recall  f1-score      support           
0              0.651020  0.625490  0.638000   510.000000            recall higher :))
1              0.639623  0.664706  0.651923   510.000000            recall lower :((
accuracy       0.645098  0.645098  0.645098     0.645098
macro avg      0.645322  0.645098  0.644962  1020.000000
weighted avg   0.645322  0.645098  0.644962  1020.000000


### Change 2: 3 epochs

{'eval_loss': 2.698315143585205,                                    loss higher :((
 'eval_f1': 0.6381347759998692,
 'eval_precision': 0.6383890597395799,
 'eval_recall': 0.638235294117647,
 'eval_accuracy': 0.638235294117647,
 'eval_runtime': 4.0599,
 'eval_samples_per_second': 251.237,
 'eval_steps_per_second': 31.528,
 'epoch': 3.0}

Report 
               precision    recall  f1-score      support
0              0.643002  0.621569  0.632104   510.000000
1              0.633776  0.654902  0.644166   510.000000
accuracy       0.638235  0.638235  0.638235     0.638235
macro avg      0.638389  0.638235  0.638135  1020.000000
weighted avg   0.638389  0.638235  0.638135  1020.000000


### Same as above, with 3 epochs, but got different results (no seed)

{'eval_loss': 0.9071153402328491,
 'eval_f1': 0.6614102997081721,
 'eval_precision': 0.6624448382501129,
 'eval_recall': 0.6617647058823529,
 'eval_accuracy': 0.6617647058823529,
 'eval_runtime': 5.1484,
 'eval_samples_per_second': 198.121,
 'eval_steps_per_second': 24.862,
 'epoch': 3.0}

               precision    recall  f1-score      support
0              0.672956  0.629412  0.650456   510.000000
1              0.651934  0.694118  0.672365   510.000000
accuracy       0.661765  0.661765  0.661765     0.661765
macro avg      0.662445  0.661765  0.661410  1020.000000
weighted avg   0.662445  0.661765  0.661410  1020.000000


### Change 3: 3 epochs + batch size = 4

{'eval_loss': 2.2720787525177,
 'eval_f1': 0.6508461538461539,
 'eval_precision': 0.6512129380053908,
 'eval_recall': 0.6509803921568627,
 'eval_accuracy': 0.6509803921568628,
 'eval_runtime': 5.1256,
 'eval_samples_per_second': 199.001,
 'eval_steps_per_second': 49.75,
 'epoch': 3.0}

Report:

              precision    recall  f1-score     support
0              0.657143  0.631373  0.644000   510.00000
1              0.645283  0.670588  0.657692   510.00000
accuracy       0.650980  0.650980  0.650980     0.65098
macro avg      0.651213  0.650980  0.650846  1020.00000
weighted avg   0.651213  0.650980  0.650846  1020.00000


### change 4: 3 epochs + batch size = 2
{'eval_loss': 3.367004632949829,
 'eval_f1': 0.6440762526206911,
 'eval_precision': 0.644184722612211,
 'eval_recall': 0.6441176470588235,
 'eval_accuracy': 0.6441176470588236,
 'eval_runtime': 6.0719,
 'eval_samples_per_second': 167.986,
 'eval_steps_per_second': 83.993,
 'epoch': 3.0}

 Report
               precision    recall  f1-score      support
0              0.641075  0.654902  0.647915   510.000000
1              0.647295  0.633333  0.640238   510.000000
accuracy       0.644118  0.644118  0.644118     0.644118
macro avg      0.644185  0.644118  0.644076  1020.000000
weighted avg   0.644185  0.644118  0.644076  1020.000000

### Change 5: 1 epoch + batch size 14


{'eval_loss': 3.2993574142456055,
 'eval_f1': 0.6330949367332099,
 'eval_precision': 0.633680769705193,
 'eval_recall': 0.6333333333333333,
 'eval_accuracy': 0.6333333333333333,
 'eval_runtime': 3.8416,
 'eval_samples_per_second': 265.511,
 'eval_steps_per_second': 19.002,
 'epoch': 1.0}

               precision    recall  f1-score      support
0              0.626866  0.658824  0.642447   510.000000
1              0.640496  0.607843  0.623742   510.000000
accuracy       0.633333  0.633333  0.633333     0.633333
macro avg      0.633681  0.633333  0.633095  1020.000000
weighted avg   0.633681  0.633333  0.633095  1020.000000


## FROM HERE WITH MORE DATA (TRAIN MERGED WITH TEST)

## One of the better: Change 6: batch size 16

{'eval_loss': 0.6043114066123962,
 'eval_f1': 0.6668140940586761,
 'eval_precision': 0.6693404634581106,
 'eval_recall': 0.6676470588235295,
 'eval_accuracy': 0.6676470588235294,
 'eval_runtime': 3.9702,
 'eval_samples_per_second': 256.916,
 'eval_steps_per_second': 16.12,
 'epoch': 1.0}

 report
              precision    recall  f1-score      support
0              0.686275  0.617647  0.650155   510.000000
1              0.652406  0.717647  0.683473   510.000000
accuracy       0.667647  0.667647  0.667647     0.667647
macro avg      0.669340  0.667647  0.666814  1020.000000
weighted avg   0.669340  0.667647  0.666814  1020.000000


## One of the better: Change 7: 2 epoch + batch size 16                         sligthly better but similar to above

{'eval_loss': 0.6960973143577576,
 'eval_f1': 0.6783707811160118,
 'eval_precision': 0.6785659320364442,
 'eval_recall': 0.6784313725490196,
 'eval_accuracy': 0.6784313725490196,
 'eval_runtime': 4.0266,
 'eval_samples_per_second': 253.316,
 'eval_steps_per_second': 15.894,
 'epoch': 2.0}

Report
              precision    recall  f1-score      support
0              0.683468  0.664706  0.673956   510.000000
1              0.673664  0.692157  0.682785   510.000000
accuracy       0.678431  0.678431  0.678431     0.678431
macro avg      0.678566  0.678431  0.678371  1020.000000
weighted avg   0.678566  0.678431  0.678371  1020.000000


### Change 8: 3 epoch + batch size 16                         (lower than above)
{'eval_loss': 1.519633173942566,
 'eval_f1': 0.6499431376300071,
 'eval_precision': 0.6500975258818686,
 'eval_recall': 0.6499999999999999,
 'eval_accuracy': 0.65,
 'eval_runtime': 4.1948,
 'eval_samples_per_second': 243.16,
 'eval_steps_per_second': 15.257,
 'epoch': 3.0}

Report:
              precision    recall  f1-score  support
0              0.646272  0.662745  0.654405   510.00
1              0.653924  0.637255  0.645482   510.00
accuracy       0.650000  0.650000  0.650000     0.65
macro avg      0.650098  0.650000  0.649943  1020.00
weighted avg   0.650098  0.650000  0.649943  1020.00

### Change 9: 3 epoch + batch size 4                        (generally lower than above)

{'eval_loss': 2.801759958267212,
 'eval_f1': 0.63722003268288,
 'eval_precision': 0.6373076923076924,
 'eval_recall': 0.6372549019607843,
 'eval_accuracy': 0.6372549019607843,
 'eval_runtime': 4.18,
 'eval_samples_per_second': 244.018,
 'eval_steps_per_second': 61.004,
 'epoch': 3.0}

 report
 precision    recall  f1-score      support
0              0.640000  0.627451  0.633663   510.000000
1              0.634615  0.647059  0.640777   510.000000
accuracy       0.637255  0.637255  0.637255     0.637255
macro avg      0.637308  0.637255  0.637220  1020.000000
weighted avg   0.637308  0.637255  0.637220  1020.000000

### change 10: 1 epochs + batch size 20


{'eval_loss': 2.4828250408172607,
 'eval_f1': 0.6411709522637795,
 'eval_precision': 0.6411851555651251,
 'eval_recall': 0.6411764705882352,
 'eval_accuracy': 0.6411764705882353,
 'eval_runtime': 3.839,
 'eval_samples_per_second': 265.695,
 'eval_steps_per_second': 13.285,
 'epoch': 1.0}

 report: 
  precision    recall  f1-score      support
0              0.640078  0.645098  0.642578   510.000000
1              0.642292  0.637255  0.639764   510.000000
accuracy       0.641176  0.641176  0.641176     0.641176
macro avg      0.641185  0.641176  0.641171  1020.000000
weighted avg   0.641185  0.641176  0.641171  1020.000000



### change 11: 1 epoch + batch size 8

{'eval_loss': 2.7987122535705566,
 'eval_f1': 0.6450966747276998,
 'eval_precision': 0.6451002706692913,
 'eval_recall': 0.6450980392156863,
 'eval_accuracy': 0.6450980392156863,
 'eval_runtime': 4.1148,
 'eval_samples_per_second': 247.883,
 'eval_steps_per_second': 31.107,
 'epoch': 1.0}

Report
              precision    recall  f1-score      support
0              0.645669  0.643137  0.644401   510.000000
1              0.644531  0.647059  0.645793   510.000000
accuracy       0.645098  0.645098  0.645098     0.645098
macro avg      0.645100  0.645098  0.645097  1020.000000
weighted avg   0.645100  0.645098  0.645097  1020.000000





Notes for self: 
#precision: true positives, predicted unhealthy for actual unhealthy, of the number of times predicted unhealthy, how many times were correct,
#recall: also true positives, predicted unhealthy for actual unhealthy but out of the total number of unhealhty, so if the number of times i predcited healthy, how many were actually unhealthy

        # so precision is "i said it was unhealthy and i was right x times", recall is "i said it was unhealthy and i was right x times but there were y other unhealthy i didn't catch"
        #both precision and recall uses true positives, but they divide by different totals

        # Example: 
        # Preceision: I predicted 4 texts as human, 3 were actually human = 3/4 correct = 0.75
        # Recall:     I predicted 4 texts as human, (3 of them might have been correct^) but there were actually 2 I missed (guessed chat which were actually also human) = 3/5 correct = 0.60  


#F1-score: if both precision and recall are high F1 will be high, if they are both low, OR if one of them are way lower than the other F1 will be low


#accuracy: of all predcitions how many were correct, so predicted human was actual human, predicted chat was actual chat
#Macro average: simple average for each of thee precision, recall, F1 for each class, so it does NOT consider if there are way more in human than in chat or vv
#weighted average: average but takes into account the total number of texts in each class

# Test

In [46]:
#load datasets as txt

#load abusive dataset
txt_abusive = data_path / "clean_dataset_abusive.txt" #loading abusive messages dataset

#load nonabusive text file
hamspam_txt = data_path / "smsspamcollection.txt"



In [59]:
#read abusive txt
df_abusive = pd.read_csv(txt_abusive, sep='\t', header=None, names=["message"], encoding="utf-8")

#giving column with label
df_abusive["labels"] = 1

#read non abusive txt
df_hamspam = pd.read_csv(hamspam_txt, sep='\t', names=["labels", "message"])

print(df_abusive.head())
print(df_hamspam.head())

                                             message  labels
0  \r\r \r\r\r\r  Hey\r  whats up. Okay don't tex...       1
1  \r\r  taking\r  a shower i wish you were here ...       1
2  \r \r\r\r  damn\r  you have the nicest boobs. ...       1
3  \r \r\r\r\t\r\t\t\r\t\t\tHey its over between ...       1
4  \r  Hey\r  can i see you fb password i feel li...       1
  labels                                            message
0    ham  Go until jurong point, crazy.. Available only ...
1    ham                      Ok lar... Joking wif u oni...
2   spam  Free entry in 2 a wkly comp to win FA Cup fina...
3    ham  U dun say so early hor... U c already then say...
4    ham  Nah I don't think he goes to usf, he lives aro...


In [60]:
#keep only ham messages
df_ham_full = df_hamspam[df_hamspam["labels"] == "ham"]

# pick 171 random ham messages
df_ham = df_ham_full.sample(n=171, random_state=1802)

print(len(df_ham))

171


In [49]:
print(df_ham)

     label                                            message
5432   ham                             Thanx a lot 4 ur help!
3226   ham                   I need... Coz i never go before 
2714   ham  Nope i'm not drivin... I neva develop da photo...
563    ham  Geeeee ... I love you so much I can barely sta...
1921   ham                      Dont know you bring some food
...    ...                                                ...
1934   ham                            R u over scratching it?
2812   ham  Thinkin about someone is all good. No drugs fo...
1728   ham                           I went to project centre
785    ham  Dont think so. It turns off like randomlly wit...
2487   ham         I dont thnk its a wrong calling between us

[171 rows x 2 columns]


In [61]:
# change ham to 0
df_ham["labels"] = df_ham["labels"].map({"ham": 0})

print(df_ham)

      labels                                            message
5432       0                             Thanx a lot 4 ur help!
3226       0                   I need... Coz i never go before 
2714       0  Nope i'm not drivin... I neva develop da photo...
563        0  Geeeee ... I love you so much I can barely sta...
1921       0                      Dont know you bring some food
...      ...                                                ...
1934       0                            R u over scratching it?
2812       0  Thinkin about someone is all good. No drugs fo...
1728       0                           I went to project centre
785        0  Dont think so. It turns off like randomlly wit...
2487       0         I dont thnk its a wrong calling between us

[171 rows x 2 columns]


In [62]:
#turn into hugging face ds
ds_abusive = Dataset.from_pandas(df_abusive) 
ds_ham = Dataset.from_pandas(df_ham) 

#removing added index column
ds_ham = ds_ham.remove_columns("__index_level_0__")

In [63]:
print(ds_abusive)
print(ds_ham)

Dataset({
    features: ['message', 'labels'],
    num_rows: 171
})
Dataset({
    features: ['labels', 'message'],
    num_rows: 171
})


In [64]:
#merge the two datasets

#checking columns names are the same 
print(ds_abusive.column_names)
print(ds_ham.column_names)

['message', 'labels']
['labels', 'message']


In [66]:
#merging the two ds
merged_test_ds = concatenate_datasets([ds_abusive, ds_ham])

#shuffle
merged_test_ds = merged_test_ds.shuffle(seed=1802)

print(merged_test_ds["message"][:5])
print(merged_test_ds["labels"][:20])


["There generally isn't one. It's an uncountable noun - u in the dictionary. pieces of research?", 'Cool, want me to go to kappa or should I meet you outside mu', 'Oh, i will get paid. The most outstanding one is for a commercial i did for Hasbro...in AUGUST! They made us jump through so many hoops to get paid. Still not.', 'Have a nice day my dear.', "Nope i'm not drivin... I neva develop da photos lei..."]
[0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0]


In [ ]:
#cleaning

#remove the beginning "" ooor make each "" one row



In [ ]:
#change column names

In [71]:
# defining column and classes
#number is defined previously

merged_test_ds = merged_test_ds.cast_column("labels", ClassLabel(num_classes=num_classes))

Casting the dataset: 100%|██████████| 342/342 [00:00<00:00, 115887.22 examples/s]


In [74]:
#defining function for the message column

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["message"], truncation=True)

In [75]:
tokenized_test = merged_test_ds.map(preprocess_function, batched=True)


Map: 100%|██████████| 342/342 [00:00<00:00, 13521.21 examples/s]


In [76]:
trainer.evaluate(tokenized_test)

/work/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'eval_loss': 0.6980554461479187,
 'eval_model_preparation_time': 0.0049,
 'eval_f1': 0.3333333333333333,
 'eval_precision': 0.25,
 'eval_recall': 0.5,
 'eval_accuracy': 0.5,
 'eval_runtime': 4.3536,
 'eval_samples_per_second': 78.555,
 'eval_steps_per_second': 5.053}

In [77]:
#make predictions on val ds
predictions = trainer.predict(tokenized_val)

# Get raw logits from trainer predictions
logits = predictions.predictions

# Convert logits to predicted class indices
y_pred = np.argmax(logits, axis=-1)

# Get true labels
y_true = predictions.label_ids

classification_report_metric = evaluate.load("bstrai/classification_report") #loading .... whatever is gonna make the report 

results = classification_report_metric.compute(references=y_true, predictions=y_pred) #taking the true and predicted values

report_df = pd.DataFrame(results).transpose()#.round(2) #turning it into a pd to print the results neatly

print(report_df)

/work/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision  recall  f1-score  support
0                  0.50     1.0  0.666667    510.0
1                  0.00     0.0  0.000000    510.0
accuracy           0.50     0.5  0.500000      0.5
macro avg          0.25     0.5  0.333333   1020.0
weighted avg       0.25     0.5  0.333333   1020.0


/work/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/work/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/work/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
#steps donw for the train and val


ds_train[:5]

# print example
print(ds_train["comment"][12]) #remember zero indexed :)
print(ds_val["comment"][1])



#checking balance for unhealthy/healhty (BEFORE downsampling)
#1 = healhty
#0 = unhealhty

print(Counter(ds_train["unhealthy"]))
print(Counter(ds_val["unhealthy"]))
